### Homework 2 - regresson

### setup

In [2]:
import pandas as pd
import numpy as np

In [8]:
!curl -O https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  620k  100  620k    0     0   547k      0  0:00:01  0:00:01 --:--:--  548k


In [9]:
# Load the full dataset
df_full = pd.read_csv('car_fuel_efficiency_2026.csv')

In [10]:
# Using only the columns specified in the homework instructions
cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df_full[cols].copy()

### Does fuel_efficiency_mpg have a long tail?

In [11]:
y = df['fuel_efficiency_mpg']

# Skewness: near 0 = symmetric, large positive/negative = long tail in that direction
print("Skewness:", y.skew())
print()

# Summary stats — comparing mean vs median is a quick symmetry check
print(y.describe())
print()
print("Mean:", y.mean())
print("Median:", y.median())

# Bin the values to inspect the shape of the distribution directly
counts, bin_edges = np.histogram(y, bins=15)
for c, e in zip(counts, bin_edges):
    print(f"{e:6.1f} | {'#' * (c // 30)} ({c})")

Skewness: 0.08080419601878173

count    10000.000000
mean        29.997700
std          2.930245
min         19.800000
25%         28.000000
50%         30.000000
75%         31.900000
max         41.200000
Name: fuel_efficiency_mpg, dtype: float64

Mean: 29.9977
Median: 30.0
  19.8 |  (6)
  21.2 | # (46)
  22.7 | #### (124)
  24.1 | ############### (464)
  25.5 | ########################### (826)
  26.9 | ################################################# (1477)
  28.4 | ########################################################## (1766)
  29.8 | ################################################################# (1957)
  31.2 | ################################################### (1554)
  32.6 | ############################## (903)
  34.1 | ################# (528)
  35.5 | ######## (255)
  36.9 | ## (68)
  38.3 |  (24)
  39.8 |  (2)


No it does not have a long tail

#### Q1 — Column with missing values

In [12]:
# Count missing values per column — only horsepower should be non-zero
print(df.isnull().sum())

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64


#### Q2 — Median horsepower


In [13]:
# Median (50th percentile) of horsepower, including the missing-value rows dropped automatically
df['horsepower'].median()

np.float64(254.0)

#### Q3 — Fill with 0 vs. fill with mean

##### Helper Functions

In [14]:
def rmse(y, y_pred):
    # Root Mean Squared Error: measures how far predictions are from actual values
    error = y_pred - y
    return np.sqrt((error ** 2).mean())

def split_data(df, seed):
    # Standard 60/20/20 train/validation/test split, exactly as taught in the lecture
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test  # remainder, to avoid rounding mismatches

    # Shuffle row indices with a fixed seed so the split is reproducible
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    # Slice the shuffled indices into three groups
    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
    df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)
    return df_train, df_val, df_test

def prepare_X(df, fillna_value):
    # Fill missing values (only 'horsepower' has any) and return a plain NumPy array
    df_num = df.copy()
    df_num = df_num.fillna(fillna_value)
    return df_num.values

def train_linear_regression_reg(X, y, r=0.0):
    # Add a column of 1s for the bias/intercept term
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    # Normal equation with L2 regularization: w = (X^T X + r*I)^-1 X^T y
    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])  # regularization term added to the diagonal
    XTX = XTX + reg
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    # Split the bias term (w0) from the feature weights (w)
    return w_full[0], w_full[1:]

In [15]:
feature_cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

# Split using seed 42, as specified in the homework
df_train, df_val, df_test = split_data(df, seed=42)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

# --- Option A: fill missing horsepower with 0 ---
X_train_0 = prepare_X(df_train[feature_cols], 0)
X_val_0 = prepare_X(df_val[feature_cols], 0)
w0, w = train_linear_regression_reg(X_train_0, y_train, r=0.0)  # no regularization
y_pred = w0 + X_val_0.dot(w)
rmse_0 = round(rmse(y_val, y_pred), 3)

# --- Option B: fill missing horsepower with the mean (computed from TRAIN only) ---
mean_hp = df_train['horsepower'].mean()  # never use val/test data to compute this — avoids leakage
X_train_mean = prepare_X(df_train[feature_cols], mean_hp)
X_val_mean = prepare_X(df_val[feature_cols], mean_hp)
w0m, wm = train_linear_regression_reg(X_train_mean, y_train, r=0.0)
y_pred_m = w0m + X_val_mean.dot(wm)
rmse_mean = round(rmse(y_val, y_pred_m), 3)

print("RMSE with 0:", rmse_0)
print("RMSE with mean:", rmse_mean)

RMSE with 0: 2.205
RMSE with mean: 2.202


#### Q4 — Regularization sweep

In [16]:
# Reuse the same seed-42 split and 0-fill from Q3 (Option A)
X_train = prepare_X(df_train[feature_cols], 0)
X_val = prepare_X(df_val[feature_cols], 0)

# Try each regularization strength and record its validation RMSE
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    print(f"r={r}: RMSE={score}")

r=0: RMSE=2.2053
r=0.01: RMSE=2.2058
r=0.1: RMSE=2.2241
r=1: RMSE=2.3492
r=5: RMSE=2.4094
r=10: RMSE=2.4195
r=100: RMSE=2.4292


#### Q5 — Seed sensitivity

In [17]:
scores = []

# Repeat the full split + train + evaluate cycle for each seed 0–9
for seed in range(10):
    df_train, df_val, df_test = split_data(df, seed=seed)
    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values

    # Fill with 0, no regularization, per the question's instructions
    X_train = prepare_X(df_train[feature_cols], 0)
    X_val = prepare_X(df_val[feature_cols], 0)
    w0, w = train_linear_regression_reg(X_train, y_train, r=0.0)
    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)
    scores.append(score)
    print(f"seed={seed}: RMSE={score:.5f}")

# Standard deviation across all 10 seeds — measures how stable the model is to the split
std = round(np.std(scores), 3)
print("std:", std)

seed=0: RMSE=2.23920
seed=1: RMSE=2.20211
seed=2: RMSE=2.16342
seed=3: RMSE=2.20436
seed=4: RMSE=2.20188
seed=5: RMSE=2.24579
seed=6: RMSE=2.26972
seed=7: RMSE=2.19079
seed=8: RMSE=2.21661
seed=9: RMSE=2.20704
std: 0.029


#### Q6 — Final test RMSE (seed 9, combined train+val, r=0.001)

In [18]:
# Split with seed 9, as specified
df_train, df_val, df_test = split_data(df, seed=9)

# Combine train and validation into one larger training set
df_train_full = pd.concat([df_train, df_val]).reset_index(drop=True)

y_train_full = df_train_full['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

# Fill missing values with 0, train with light regularization (r=0.001)
X_train_full = prepare_X(df_train_full[feature_cols], 0)
X_test = prepare_X(df_test[feature_cols], 0)

w0, w = train_linear_regression_reg(X_train_full, y_train_full, r=0.001)
y_pred = w0 + X_test.dot(w)

# Evaluate on the held-out test set — the only time test data is used
score = rmse(y_test, y_pred)
print("Test RMSE:", score, "rounded:", round(score, 3))

Test RMSE: 2.2358277471945915 rounded: 2.236
